# 04 — Deep Learning: LSTM & GRU Traffic Forecasting
## GeoMind AI | Amazon Applied Scientist I Intern

---

**Objective:** Train and compare deep sequential architectures (LSTM, GRU) for next-hour urban traffic volume forecasting. Investigate context window ablation (L = 6, 12, 24).

**Demonstrated Skills:**
- Temporal sequence modeling with sliding windows
- PyTorch training loop with early stopping & LR scheduling
- Huber loss for outlier robustness
- Target normalization & inverse scaling
- Controlled ablation experiments
- Experiment registry tracking


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_preprocessing import prepare_datasets
from src.train_dl import TrafficLSTM, TrafficGRU, build_sequences
from src.evaluate import compute_metrics

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | Device: {DEVICE.upper()}")


## 1. Load Preprocessed Datasets

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test, feat_names = prepare_datasets()
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Features: {len(feat_names)}")


## 2. Target Normalization

StandardScaler applied to targets separately from features.
Predictions are inverse-transformed back to vehicle counts for evaluation.


In [ ]:
from sklearn.preprocessing import StandardScaler

target_scaler = StandardScaler()
y_train_scaled = target_scaler.fit_transform(y_train.reshape(-1,1)).flatten()
y_val_scaled   = target_scaler.transform(y_val.reshape(-1,1)).flatten()
y_test_scaled  = target_scaler.transform(y_test.reshape(-1,1)).flatten()

print(f"Target scaler → mean={target_scaler.mean_[0]:.1f}, scale={target_scaler.scale_[0]:.1f}")


## 3. Model Architecture

**LSTM**: 2-layer with dropout → FC head (64→32→1)  
**GRU**: Same structure with GRU cells (~25% fewer parameters)

Both use: HuberLoss, AdamW (lr=1e-3, wd=1e-4), gradient clipping (max_norm=1.0)


In [ ]:
INPUT_DIM = X_train.shape[1]
lstm = TrafficLSTM(input_dim=INPUT_DIM, hidden_dim=64, num_layers=2, dropout=0.2)
gru  = TrafficGRU(input_dim=INPUT_DIM, hidden_dim=64, num_layers=2, dropout=0.2)

def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"LSTM parameters: {n_params(lstm):,}")
print(f"GRU  parameters: {n_params(gru):,}")
print(f"GRU savings    : {(1 - n_params(gru)/n_params(lstm))*100:.1f}%")


## 4. Load Pre-Trained Results from Experiment Registry

In [ ]:
results_df = pd.read_csv('experiments/results.csv')
dl_df = results_df[results_df['model_family'] == 'Deep Learning'].copy()
print(f"Loaded {len(dl_df)} DL experiment records")
print()

display_cols = ['model_name', 'sequence_length', 'val_mae', 'val_r2', 'test_mae', 'test_r2', 'training_time_sec']
print(dl_df[display_cols].to_string(index=False))


## 5. Context Length Ablation: L ∈ {6, 12, 24}

In [ ]:
lstm_df = dl_df[dl_df['model_name'].str.contains('LSTM')].sort_values('sequence_length')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')
for ax in axes:
    ax.set_facecolor('#1a1d27')

ax = axes[0]
ax.plot(lstm_df['sequence_length'], lstm_df['val_mae'],  'o-', color='#5bc8f5', lw=2.5, ms=9, label='Val MAE')
ax.plot(lstm_df['sequence_length'], lstm_df['test_mae'], 's--',color='#ff8844', lw=2,   ms=7, label='Test MAE')
for _, r in lstm_df.iterrows():
    ax.annotate(f"{r['val_mae']:.0f}", (r['sequence_length'], r['val_mae']),
                textcoords='offset points', xytext=(0,10), color='white', fontsize=9, ha='center')
ax.set_xlabel('Context Length L (hours)', color='white', fontsize=11)
ax.set_ylabel('MAE (vehicles/hr)',        color='white', fontsize=11)
ax.set_title('LSTM: Context Length vs MAE', color='white', fontsize=12, fontweight='bold')
ax.set_xticks([6, 12, 24])
ax.tick_params(colors='white')
ax.legend(facecolor='#1a1d27', labelcolor='white')
ax.spines[['top','right']].set_visible(False)
ax.spines[['bottom','left']].set_color('#444')

ax = axes[1]
ax.plot(lstm_df['sequence_length'], lstm_df['val_r2'],  'o-', color='#44ff88', lw=2.5, ms=9, label='Val R²')
ax.plot(lstm_df['sequence_length'], lstm_df['test_r2'], 's--',color='#ff6b6b', lw=2,   ms=7, label='Test R²')
for _, r in lstm_df.iterrows():
    ax.annotate(f"{r['val_r2']:.4f}", (r['sequence_length'], r['val_r2']),
                textcoords='offset points', xytext=(0,8), color='white', fontsize=9, ha='center')
ax.set_xlabel('Context Length L (hours)', color='white', fontsize=11)
ax.set_ylabel('R² Score',                 color='white', fontsize=11)
ax.set_title('LSTM: Context Length vs R²', color='white', fontsize=12, fontweight='bold')
ax.set_xticks([6, 12, 24])
ax.tick_params(colors='white')
ax.legend(facecolor='#1a1d27', labelcolor='white')
ax.spines[['top','right']].set_visible(False)
ax.spines[['bottom','left']].set_color('#444')

plt.suptitle('GeoMind AI — LSTM Context Length Ablation', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
Path('docs/figures').mkdir(parents=True, exist_ok=True)
plt.savefig('docs/figures/dl_context_ablation.png', dpi=130, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("Finding: L=6 achieves lowest Val MAE. Shorter context avoids noise accumulation.")


## 6. LSTM vs GRU (Context L=12)

In [ ]:
l12 = dl_df[dl_df['sequence_length'] == 12].copy()

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')

models_names = l12['model_name'].tolist()
x = range(len(models_names))
vals  = l12['val_mae'].tolist()
tests = l12['test_mae'].tolist()
width = 0.35

b1 = ax.bar([xi - width/2 for xi in x], vals,  width, label='Val MAE',  color='#5bc8f5', alpha=0.85)
b2 = ax.bar([xi + width/2 for xi in x], tests, width, label='Test MAE', color='#7c5cbf', alpha=0.85)
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 2,
            f"{b.get_height():.0f}", ha='center', va='bottom', color='white', fontsize=10)

ax.set_xticks(list(x))
ax.set_xticklabels(models_names, color='white', fontsize=10)
ax.set_ylabel('MAE (vehicles/hr)', color='white', fontsize=11)
ax.set_title('LSTM vs GRU (L=12): Val & Test MAE Comparison', color='white', fontsize=12, fontweight='bold')
ax.tick_params(colors='white')
ax.legend(facecolor='#1a1d27', labelcolor='white')
ax.spines[['top','right']].set_visible(False)
ax.spines[['bottom','left']].set_color('#444')
plt.tight_layout()
plt.savefig('docs/figures/lstm_vs_gru.png', dpi=130, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()


## 7. Key Findings

| Model | L | Val MAE | Test MAE | Test R² | Train Time |
|-------|---|---------|----------|---------|------------|
| LSTM (L=6)  | 6  | ~221 | ~212 | ~0.969 | ~27s |
| LSTM (L=24) | 24 | ~226 | ~223 | ~0.969 | ~65s |
| GRU (L=12)  | 12 | ~235 | ~224 | ~0.971 | ~58s |
| LSTM (L=12) | 12 | ~239 | ~228 | ~0.965 | ~39s |

**Key Insights:**
1. **L=6 wins** on this dataset — shorter context avoids noise from less-predictive historical hours  
2. **GRU is competitive** with LSTM at L=12 with 25% fewer parameters  
3. **DL underperforms XGBoost** — engineered lag features already capture temporal structure  
4. **DL advantage**: no manual lag feature engineering required for raw input spaces  

**→ Next: `05_error_analysis.ipynb` — Deep residual diagnostics and SHAP interpretability**
